In [25]:
import pandas as pd
import numpy as np
import re
import pdb
from collections import defaultdict
import os

In [26]:
# Einlesen der Date aus dem Notebook "1_PuRe Rohdatenextrakt.jpynb"
df_pure = pd.read_excel('Daten/df_pure.xlsx')

# Alternativ kann ich auch direkt die Daten aus dem json-download über Openrefine einlesen
#df_pure = pd.read_excel('download-json.xlsx')

# Zur Sicherheit die Anzahl an IDs = Datensätzen zählen
df_pure['90999_ItemID'].nunique() 

447

In [27]:
# Auskommentierung aufheben, um es sich anzusehen
df_pure

,90999_ItemID,data - versionNumber,data - modificationDate,data - versionState,data - versionPid,data - lastModificationDate,data - publicState,data - objectPid,data - creationDate,data - message,...,data - latestVersion - objectId,data - latestVersion - versionNumber,data - latestVersion - modificationDate,data - latestVersion - versionState,data - latestVersion - versionPid,data - latestVersion - modifier - objectId,Item_seq,Name,kleinster_wert,1103_Ersterscheinungsdatum
0,item_3699285,2.0,2026-04-23T12:29:32.939+00:00,RELEASED,hdl:21.11116/0000-0012-FA72-2,2026-04-23T12:29:32.939+00:00,RELEASED,hdl:21.11116/0000-0012-BC65-7,2026-03-25T10:54:08.925+00:00,NaN,...,item_3699285,2.0,2026-04-23T12:29:32.939+00:00,RELEASED,hdl:21.11116/0000-0012-FA72-2,user_1241569,1,Luca Cigna,2026.0,2026.0
1,item_3699285,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,2,NaN,NaN,NaN
2,item_3699285,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,3,NaN,NaN,NaN
3,item_3699285,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,4,Nina Lopez-Uroz,NaN,NaN
4,item_3699285,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,5,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2332,item_3458140,2.0,2022-11-09T13:27:47.664+00:00,RELEASED,hdl:21.11116/0000-000B-6898-5,2022-11-09T13:27:47.664+00:00,RELEASED,hdl:21.11116/0000-000B-5C0E-0,2022-11-02T13:32:56.911+00:00,NaN,...,item_3458140,2.0,2022-11-09T13:27:47.664+00:00,RELEASED,hdl:21.11116/0000-000B-6898-5,user_1241569,1,Vincenzo Maccarrone,2022.0,2022.0
2333,item_3458140,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,2,Arianna Tassinari,NaN,NaN
2334,item_3458140,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,3,NaN,NaN,NaN
2335,item_3458140,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,4,NaN,NaN,NaN


In [28]:
# Umbenennung der hier relevanten Spalten
df_pure = df_pure.rename(columns={
    'data - metadata - title':'1001_Titel',
    'data - metadata - genre':'1002_Genre',
    'data - metadata - degree' : '1021_Degree',
    'data - metadata - sources - _ - title':'1101_Quellentitel',
    'data - metadata - sources - _ - genre':'1100_Quellen-Genre',
    'data - metadata - reviewMethod':'1108_Review',
    '1103_Ersterscheinungsdatum':'Datum',
    'data - objectId':'90999_ItemID',}) 




In [29]:
# Datumsspalte bereinigen, da durch Excel-Export verändert und Werte auffüllen
df_pure['Datum'] = df_pure['Datum'].astype(str).str.extract(r'(\d{4})')
df_pure['Datum'] = df_pure['Datum'].ffill() 

# Spalteninhalt wieder als Zahl definieren
df_pure['Datum'] = df_pure['Datum'].astype(int)

# Blick auf die enthaltenen Daten 
df_pure['Datum'].unique()

array([2026, 2025, 2024, 2023, 2017, 2022, 2020, 2021])

# Reduzierung auf zu betrachtende Jahre - falls Gesamtüberblick gewünscht ist Zeile auskommentieren oder nicht laufen lassen

In [30]:
df_years = df_pure

years = [2023, 2024, 2025]   # oder bei einem Einzahljahr so [2025]

df_pure

,90999_ItemID,data - versionNumber,data - modificationDate,data - versionState,data - versionPid,data - lastModificationDate,data - publicState,data - objectPid,data - creationDate,data - message,...,data - latestVersion - objectId,data - latestVersion - versionNumber,data - latestVersion - modificationDate,data - latestVersion - versionState,data - latestVersion - versionPid,data - latestVersion - modifier - objectId,Item_seq,Name,kleinster_wert,Datum
0,item_3699285,2.0,2026-04-23T12:29:32.939+00:00,RELEASED,hdl:21.11116/0000-0012-FA72-2,2026-04-23T12:29:32.939+00:00,RELEASED,hdl:21.11116/0000-0012-BC65-7,2026-03-25T10:54:08.925+00:00,NaN,...,item_3699285,2.0,2026-04-23T12:29:32.939+00:00,RELEASED,hdl:21.11116/0000-0012-FA72-2,user_1241569,1,Luca Cigna,2026.0,2026
1,item_3699285,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,2,NaN,NaN,2026
2,item_3699285,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,3,NaN,NaN,2026
3,item_3699285,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,4,Nina Lopez-Uroz,NaN,2026
4,item_3699285,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,5,NaN,NaN,2026
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2332,item_3458140,2.0,2022-11-09T13:27:47.664+00:00,RELEASED,hdl:21.11116/0000-000B-6898-5,2022-11-09T13:27:47.664+00:00,RELEASED,hdl:21.11116/0000-000B-5C0E-0,2022-11-02T13:32:56.911+00:00,NaN,...,item_3458140,2.0,2022-11-09T13:27:47.664+00:00,RELEASED,hdl:21.11116/0000-000B-6898-5,user_1241569,1,Vincenzo Maccarrone,2022.0,2022
2333,item_3458140,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,2,Arianna Tassinari,NaN,2022
2334,item_3458140,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,3,NaN,NaN,2022
2335,item_3458140,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,4,NaN,NaN,2022


# Ermittlung von Zeitschriftentiteln, in denen veröffentlicht wurde

In [31]:
# filter the DataFrame to only include rows where "Quellen-genre" is "Journal"
journal_df = df_pure[df_pure['1100_Quellen-Genre'] == 'JOURNAL']

# count the occurrences of each unique value in the "quellentitel" column
quellentitel_counts = journal_df['1101_Quellentitel'].value_counts()

# create a new DataFrame with the counts
quellentitel_counts_df = pd.DataFrame({'1101_Quellentitel': quellentitel_counts.index, 'count': quellentitel_counts.values})

# export the results to excel
quellentitel_counts_df.to_excel('Results/Zeitschriftentitel_nach_Haeufigkeit.xlsx', index=False)

# print the new DataFrame
print(quellentitel_counts_df)

                                     1101_Quellentitel  count
0                                Socio-Economic Review     23
1                    Journal of European Public Policy     12
2    Economic Sociology: Perspectives and Conversat...     11
3            Review of International Political Economy     10
4                                 Competition & Change      9
..                                                 ...    ...
112                            Politics and Governance      1
113                                Global Perspectives      1
114                             Frontiers in Sociology      1
115            Structural Change and Economic Dynamics      1
116                             West European Politics      1

[117 rows x 2 columns]


In [33]:
# Übersicht der unterschiedlichen Genres, in denen veröffentlicht wird

# Zählung der unterschiedlichen Genre
genre_counts = df_pure['1002_Genre'].value_counts()

# create a new DataFrame with the counts
genre_counts_df = pd.DataFrame({'0002_Genre': genre_counts.index, 'Zaehler': genre_counts.values})

# neue Spalte mit den Prozentwerden hinzufügen
total_sum_genre = genre_counts_df['Zaehler'].sum()

# Prozentsatz berechnen und in neuer Spalte 'Prozent' speichern
genre_counts_df['Prozent'] = (genre_counts_df['Zaehler'] / total_sum_genre) * 100

# export the results to excel
genre_counts_df.to_excel('Results/Bericht_Genreliste_nach_Häufigkeit_inkl_Prozente.xlsx', index=False)

# Ergebnis hier anzeigen
print(genre_counts_df)

                           0002_Genre  Zaehler    Prozent
0                             ARTICLE     1236  52.888318
1                               PAPER      194   8.301241
2   CONTRIBUTION_TO_COLLECTED_EDITION      176   7.531023
3                           BLOG_POST      117   5.006418
4                    DATA_PUBLICATION      115   4.920839
5                              THESIS      110   4.706889
6                                BOOK       77   3.294822
7                         BOOK_REVIEW       76   3.252033
8            CONTRIBUTION_TO_HANDBOOK       57   2.439024
9                            PREPRINT       50   2.139495
10                          EDITORIAL       40   1.711596
11                              ISSUE       35   1.497647
12                  COLLECTED_EDITION       32   1.369277
13       CONTRIBUTION_TO_ENCYCLOPEDIA       14   0.599059
14                           HANDBOOK        5   0.213950
15                          BOOK_ITEM        3   0.128370
